# Adjoint Sensitivity Method

A personal reference for the adjoint method: what it is, why it saves memory, how it works on ODEs, and when it breaks. Every claim is verified with code. Only `numpy` is needed.

**Prerequisite:** [automatic_differentiation.ipynb](automatic_differentiation.ipynb) — specifically Section 4 (backprop by hand on 2 Euler steps). This notebook picks up from that example.

**Contents**
1. The big picture in five lines
2. Notation for the toy model
3. The adjoint trick (same 2-step example, no tape)
4. The continuous adjoint method
5. Where the minus sign in dλ/dt comes from
6. What ∂f/∂x means (and what it is not)
7. Reading the gradient formula: λ · ∂f/∂w
8. Why there's no separate "acceleration path"
9. Bigger example: two states, two weights
10. Real neural ODEs: vector-Jacobian products
11. When the adjoint breaks: backward instability
12. When can states be rebuilt backward? (reversibility)
13. Neural ODE "depth": shared weights and solver steps
14. Training loop vocabulary: iteration, batch, epoch
15. Practical notes for a neural ODE project
16. Quick self-test

## 1. The big picture in five lines

- **Optimizers (SGD, Adam) don't use the chain rule.** They only consume a gradient. The chain rule is how the gradient is *computed* (backprop = reverse-mode automatic differentiation).
- **Backprop stores** every intermediate state during the forward pass so the backward pass can use it. Memory grows with the number of steps/layers.
- **The adjoint method recomputes** the intermediate states by running the dynamics backward, so it only keeps the final state. Memory is constant in the number of solver steps (O(1) in depth, not O(1) overall).
- It only works when the computation is a **continuous, reversible evolution** (an ODE) that can be integrated backward stably.
- **PyTorch's default is still backprop** (store everything). The adjoint is opt-in for ODE models, e.g. `torchdiffeq.odeint_adjoint`, and it still uses AD internally.

## 2. Notation for the toy model

| Symbol | Meaning | Value used |
|---|---|---|
| x(t) | state (a single number) | x(0) = 1 |
| w | the only trainable parameter ("weight") | w = 0.5 |
| T | end time | T = 1 |
| L | loss | L = x(T) |
| λ(t) | **adjoint** = ∂L/∂x(t): how much the loss changes if the state at time t is nudged | |

Model: **dx/dt = f(x, w) = w · x**

Partial derivatives used everywhere:
- ∂f/∂x = w (how the speed reacts to the state)
- ∂f/∂w = x (how the speed reacts to the weight)

Exact answer to aim for: x(T) = e^{wT}, so **dL/dw = T · e^{wT} = 1.6487**.

Backprop for this model is worked out in full in [automatic_differentiation.ipynb §4](automatic_differentiation.ipynb). That section is the starting point here.

## 3. The adjoint trick (same 2-step example, no tape)

Backprop stored all of [x₀, x₁, x₂]. The adjoint trick: **store only x₂ = 1.5625** and rebuild earlier states on demand by running the step backward:

x₁ = x₂ / (1 + hw) = 1.25, then x₀ = x₁ / (1 + hw) = 1.0

Everything else (the λ recurrence, the gradient accumulation) is identical to backprop. The gradient is still 1.25.

> **The whole idea in one sentence:** earlier states are *recomputed instead of remembered*. Everything that follows is the continuous-time version of this.

In [ ]:
import numpy as np

w, h = 0.5, 0.5

# Forward pass: Euler x2 steps, but store ONLY the final state
x = 1.0
for _ in range(2):
    x = x + h * w * x
x2_stored = x   # = 1.5625, the only thing we keep

# Backward pass: rebuild each earlier state as needed, then apply the same rule as backprop
x = x2_stored
lam, grad = 1.0, 0.0
for _ in range(2):
    x_prev = x / (1 + h * w)        # run the Euler step backward to recover x_k from x_{k+1}
    grad += lam * h * x_prev        # same contribution as in backprop
    lam  *= (1 + h * w)             # same lambda recurrence as in backprop
    x = x_prev

print(f"dL/dw (adjoint, only x2 stored) = {grad:.4f}")
print(f"rebuilt x0 = {x:.4f}   (should be 1.0)")

## 4. The continuous adjoint method

Forward: solve dx/dt = f from 0 to T. **Store only x(T).**

Backward: integrate three things together from T down to 0:

| Quantity | Backward equation | Value at t = T |
|---|---|---|
| state x | dx/dt = f(x, w) (run in reverse) | x(T) |
| adjoint λ | dλ/dt = −λ · ∂f/∂x | ∂L/∂x(T) |
| gradient g | accumulate λ · ∂f/∂w | 0 |

At t = 0: g = dL/dw and λ(0) = ∂L/∂x(0).

Each row is the continuous form of something in Section 3:
- x row = recomputing x₁, x₀ by running the step backward
- λ row = "λ₁ = λ₂ × 1.25" — the recurrence run continuously
- g row = summing the per-step gradient contributions

**Toy model values** (λ(t) = e^{w(1−t)}, x(t) = e^{wt}, and λ·x = e^{w} is constant):

| t | x(t) rebuilt | λ(t) | g so far |
|---|---|---|---|
| 1.0 | 1.6487 | 1.0000 | 0 |
| 0.5 | 1.2840 | 1.2840 | 0.8244 |
| 0.0 | 1.0000 | 1.6487 | **1.6487** |

Memory during the backward pass: x, λ, g — 3 numbers, regardless of how many solver steps were taken.

In [ ]:
def adjoint_toy(w, T=1.0, n=100_000):
    x = np.exp(w * T)           # forward: keep only x(T)
    lam, g, dt = 1.0, 0.0, T / n
    for _ in range(n):          # integrate backward from T to 0
        g   += dt * lam * x     # accumulate lambda * df/dw  (df/dw = x for this model)
        lam += dt * lam * w     # d lambda/dt = -w*lambda → backward Euler: lam += dt*w*lam
        x   -= dt * w * x       # rebuild x backward
    return g, lam, x

g, lam0, x_rebuilt = adjoint_toy(0.5)
print(f"dL/dw  adjoint = {g:.4f}   exact = {np.exp(0.5):.4f}")
print(f"lambda(0)      = {lam0:.4f}   (= dL/dx(0) = e^0.5)")
print(f"rebuilt x(0)   = {x_rebuilt:.4f}")

## 5. Where the minus sign in dλ/dt comes from

**Common mistake:** writing ∂L/∂x(t) = ∂L/∂f · ∂f/∂x and asking why ∂L/∂f = −λ. It doesn't. The state x(t) affects L through the **next state**, not "through f" in a single product:

x(t) → x(t + ε) → … → x(T) → L

Deriving the equation step by step:
1. One small step: x(t + ε) = x(t) + ε · f(x(t), w)
2. Differentiate w.r.t. x(t): ∂x(t + ε)/∂x(t) = 1 + ε · ∂f/∂x
3. Chain rule: λ(t) = λ(t + ε) · (1 + ε · ∂f/∂x)
4. Rearrange: [λ(t + ε) − λ(t)] / ε = **−** λ(t + ε) · ∂f/∂x
5. Let ε → 0: **dλ/dt = −λ · ∂f/∂x**

The minus sign appears because the chain rule gives "λ(t) = λ(t+ε) + something", while d/dt is defined as "later − earlier" / ε.

Numeric check (discrete, h = 0.5): (λ₂ − λ₁)/h = (1 − 1.25)/0.5 = −0.5 = −w·λ₂ ✓

**Intuition:** reading backward in time, λ grows (1 → 1.28 → 1.65). An earlier nudge has longer to be amplified by the dynamics, so it matters more. Reading forward in time, λ shrinks → dλ/dt < 0.

## 6. What ∂f/∂x means (and what it is not)

∂f/∂x is **not** a second time derivative. It asks: *freeze time; compare two nearby states; how different are their speeds?*

- f = w·x, w = 0.5: at x = 2.0 the speed is 1.00, at x = 2.1 it is 1.05 → ∂f/∂x ≈ 0.05/0.1 = 0.5 = w
- **d²x/dt²** (acceleration) is a different quantity, but related: d²x/dt² = ∂f/∂x · f (+ ∂f/∂t). For the toy model: w·(w·x) = w²x.

**Geometric meaning:** whether nearby trajectories spread apart or pull together.
- ∂f/∂x > 0: gaps grow → early nudges amplify → λ grows when traced backward.
- ∂f/∂x < 0: gaps shrink forward → they **grow when run backward** → instability (Section 11).

For a vector state, the Jacobian entry (i, j) = how the speed of xᵢ depends on the value of xⱼ.

## 7. Reading the gradient formula: λ · ∂f/∂w

$$\frac{dL}{dw} = \int_0^T \lambda(t)\,\frac{\partial f}{\partial w}\,dt$$

Story for a nudge δw at each moment:
1. The weight changes the speed: by (∂f/∂w)·δw.
2. Over dt this shifts the state by (∂f/∂w)·δw·dt.
3. That state shift changes the loss by λ(t)·(∂f/∂w)·δw·dt.

Add up over all t. The weight acts at *every* moment; λ says what each moment's effect is worth at the end.

**Not to confuse with λ · f** ("loss sensitivity × how fast x moves"). That is the **Hamiltonian**: it equals dL/dT, the effect of running the ODE a little longer. For the toy model λ·f = 0.5·e^{0.5} ≈ 0.824 = d/dT e^{0.5T} at T = 1, and it is constant in time when f has no explicit t.

In [ ]:
import numpy as np

w, T = 0.5, 1.0
t = np.linspace(0, T, 5)
lam = np.exp(w * (T - t))   # lambda(t) = e^{w(T-t)}
x   = np.exp(w * t)         # x(t) = e^{wt}

print("lambda * df/dw  (integrand of dL/dw):", np.round(lam * x, 4))   # constant = e^w
print("lambda * f      (Hamiltonian = dL/dT):", np.round(lam * w * x, 4), "  dL/dT =", round(w * np.exp(w * T), 4))

## 8. Why there's no separate "acceleration path"

Question: why only "weight → speed"? Why not "weight → acceleration → speed → position → loss"?

- In dx/dt = f(x, w), **w appears only in the speed formula**. Acceleration is a *consequence* (d²x/dt² = ∂f/∂x · f). A separate w → acceleration path would count the same effect twice.
- The knock-on effects (changed state → changed speed later → …) are **already inside λ**. That's what the ∂f/∂x term in dλ/dt carries. In the toy model λ(0) = 1.65, not 1: the extra 0.65 is the accumulated knock-on effect.
- Within a tiny dt, the weight's acceleration-type effect on position is O(dt²). It isn't lost; it appears at *later* times and λ counts it there.

**If the model really is second-order**, e.g. d²x/dt² = F(x, v, w), make velocity part of the state:

dx/dt = v, dv/dt = F(x, v, w)

Adjoint equations (λₓ = ∂L/∂x, λᵥ = ∂L/∂v):
- dλₓ/dt = −λᵥ · ∂F/∂x
- dλᵥ/dt = −λₓ − λᵥ · ∂F/∂v
- dL/dw = ∫ λᵥ · ∂F/∂w dt

The chain appears explicitly: ∂F/∂w (w → acceleration), λᵥ (acceleration → speed), −λₓ term (speed → location), λₓ (location → loss).

**Rule:** the weight enters through whatever is being integrated; λ carries everything downstream.

## 9. Bigger example: two states, two weights

x₁ grows exponentially; x₂ is a running total of x₁:

dx₁/dt = a·x₁, dx₂/dt = b·x₁, with x(0) = (1, 0), a = 1, b = 2, T = 1, L = x₂(T)

- Jacobian ∂f/∂x = [[a, 0], [b, 0]]; ∂f/∂a = (x₁, 0); ∂f/∂b = (0, x₁)
- Adjoint (row vector): dλ/dt = −λᵀ · ∂f/∂x → dλ₁/dt = −(a·λ₁ + b·λ₂), dλ₂/dt = 0
- Start: λ(T) = (0, 1) → λ₂ ≡ 1, λ₁(t) = (b/a)(e^{a(1−t)} − 1)
- Gradients: dL/da = ∫ λ₁·x₁ dt = **2.0000**, dL/db = ∫ λ₂·x₁ dt = e − 1 = **1.7183**
- Check: L = (b/a)(e^{a} − 1) → same values ✓

Reading λ₁: at the end, nudging x₁ does nothing (no time left for x₂ to collect it); at the start, a nudge is worth a lot.

**Loss at several times** (e.g. observations at multiple frames): λ **jumps** at each observation time tₖ when passing it backward: λ(tₖ⁻) = λ(tₖ⁺) + ∂Lₖ/∂x(tₖ). Same as backprop summing gradients from multiple outputs.

In [ ]:
def two_state_adjoint(a, b, T=1.0, n=200_000):
    x = np.array([np.exp(a * T), (b / a) * (np.exp(a * T) - 1)])   # only x(T) stored
    lam = np.array([0.0, 1.0])                                     # dL/dx(T) for L = x2(T)
    ga = gb = 0.0
    dt = T / n
    for _ in range(n):
        J = np.array([[a, 0.0], [b, 0.0]])                         # df/dx Jacobian
        f = np.array([a * x[0], b * x[0]])
        ga += dt * lam @ np.array([x[0], 0.0])                     # lambda^T df/da
        gb += dt * lam @ np.array([0.0, x[0]])                     # lambda^T df/db
        lam = lam + dt * (lam @ J)                                 # backward step: d lambda/dt = -lambda^T J
        x   = x - dt * f                                           # rebuild state backward
    return ga, gb, x

a, b = 1.0, 2.0
ga, gb, x0_rebuilt = two_state_adjoint(a, b)
exact_da = b * (np.exp(a) / a - (np.exp(a) - 1) / a**2)
exact_db = (np.exp(a) - 1) / a
print(f"dL/da adjoint {ga:.4f}  exact {exact_da:.4f}")
print(f"dL/db adjoint {gb:.4f}  exact {exact_db:.4f}")
print("rebuilt x(0):", np.round(x0_rebuilt, 4))

## 10. Real neural ODEs: vector-Jacobian products

dx/dt = f(x, t; θ) with f a neural network, x of size d (e.g. 64), θ with e.g. 100k entries.

Same recipe, bigger objects. ∂f/∂θ would be 64 × 100,000 — far too big to build. But you never need it: only **λᵀ·∂f/∂x and λᵀ·∂f/∂θ** (vector-Jacobian products, VJPs). Reverse-mode AD gives both in one cheap backward pass through f at a single time point.

```python
x, lam, g = x_T, dL_dxT, zeros_like(theta)
for each solver step from T back to 0:
    fx = f(x, t, theta)
    vjp_x, vjp_theta = autograd.grad(fx, (x, theta), grad_outputs=lam)
    x   = x   - h * fx          # rebuild state backward
    lam = lam + h * vjp_x       # d lambda/dt = -lambda^T df/dx
    g   = g   + h * vjp_theta   # accumulate lambda^T df/dtheta
return g                        # = dL/dtheta
```

The autograd graph of one f call is freed after each step → memory is constant in the number of steps.

> **Key point:** the adjoint method **uses AD inside it**. It isn't an alternative to AD; it's a way of organizing AD for the operation "solve an ODE".

## 11. When the adjoint breaks: backward instability

Take **w = −5** (fast decay — like viscosity or diffusion). Euler with h = 0.1: factor per step = 1 + hw = 0.5, so x halves each step.

Running the step backward means **dividing by 0.5 = doubling** each step. A tiny error δ in the stored x(T) grows to ≈ 2¹⁰·δ ≈ 1000δ after 10 steps, and ≈ 10¹⁵·δ after 50.

With w = +0.5 the backward factor is 1/1.25 = 0.8, which *shrinks* errors: safe.

| Forward dynamics | Backward reconstruction |
|---|---|
| grow or oscillate | stable → adjoint works well |
| decay / dissipate / stiff | amplifies errors → use backprop or checkpointed adjoint |

Other costs of the continuous adjoint:
- **Slower:** typically 2–5× more evaluations of f than a plain forward solve.
- **Optimize-then-discretize mismatch:** the adjoint gives the gradient of the ideal ODE, not exactly of what the solver computed. Backprop through the solver gives the exact gradient of what was run.
- **Fixes:** checkpointed / interpolated adjoints (ACA), reversible or symplectic solvers (MALI).

In [ ]:
import numpy as np

def backward_error(w, h, steps, delta=1e-10):
    factor = 1 + h * w
    xT = factor ** steps       # true x(T) starting from x(0) = 1
    x = xT + delta             # stored x(T) with a tiny perturbation
    for _ in range(steps):
        x /= factor            # run Euler backward
    return abs(x - 1.0)        # how far from the true x(0) = 1

for w_, steps in [(0.5, 10), (-5, 10), (-5, 50)]:
    err = backward_error(w_, 0.1, steps)
    print(f"w={w_:>4}, {steps:>2} steps: error in rebuilt x(0) = {err:.3e}  (start error 1e-10)")

## 12. When can states be rebuilt backward? (reversibility)

The adjoint's memory trick needs **the previous state from the next one**. Three conditions for the evolving part:

1. **Same size throughout.** An ODE requires it (dx/dt has the shape of x). A 5 → 64 layer can't be inside the flow.
2. **Residual form: add a change, don't replace the state.** x ← x + h·f(x) can be undone; x ← ReLU(W·x) can't (negatives → 0; W may be singular).
3. **Small or controlled changes.** F(x) = −x gives x + F(x) = 0 for every x (irreversible), but x + h·F(x) = (1−h)·x is fine for h < 1. The continuous neural ODE is reversible in principle for any Lipschitz f.

**ReLU is not the problem.** You never invert f, only the step:

| xₖ | ReLU(xₖ) | xₖ + 0.1·ReLU(xₖ) |
|---|---|---|
| −3 | 0 | −3.0 (distinct) |
| −2 | 0 | −2.0 (distinct) |
| 2 | 2 | 2.2 |

**An invertible activation is neither necessary nor sufficient:** tanh(5) ≈ tanh(6) in float32, and W may not be invertible regardless.

**Related architectures:**
- **ResNet:** xₖ₊₁ = xₖ + F(xₖ) — one Euler step with h = 1. Neural ODEs came from this observation.
- **i-ResNet:** limits F's Lipschitz constant below 1; each block invertible by fixed-point iteration.
- **RevNet:** y₁ = x₁ + F(x₂), y₂ = x₂ + G(y₁); invertible with arbitrary F, G (ReLU fine — neither is ever inverted).

**Handling size changes:** encoder → **ODE block** (same size throughout) → decoder. Backprop for encoder/decoder; adjoint optionally for the ODE block. *Augmented neural ODE*: pad the input with zeros to raise the dimension.

**PDEs are fine too:** discretize space → big ODE system in time (method of lines). Weather 4D-Var and CFD shape optimization use adjoints of PDEs. The key requirement is a reversible continuous evolution in *some* variable, not specifically physical time.

In [ ]:
import numpy as np

x = np.array([-3.0, -2.0, 2.0, 3.0])
relu = np.maximum(x, 0)
step = x + 0.1 * relu                             # the Euler step x + h*f(x)
inv  = np.where(step >= 0, step / 1.1, step)      # invert the *step*, not ReLU

print("ReLU alone      :", relu, "  <- -3 and -2 both map to 0 (information lost)")
print("x + 0.1*ReLU(x) :", step, "  <- distinct")
print("recovered x     :", inv)
print()
print("tanh(5) vs tanh(6) in float32:",
      np.tanh(np.float32(5)), np.tanh(np.float32(6)),
      "  <- invertible in principle, nearly identical in practice")

## 13. Neural ODE "depth": shared weights and solver steps

Two different things both get called "layers":
1. **The network f itself:** a small ordinary network (e.g. 3-layer MLP), with discrete layers as usual.
2. **The solver's repeated calls of f:** these play the role of a ResNet's blocks.

| | Each update | Weights | Number of updates |
|---|---|---|---|
| Ordinary deep net | xₖ₊₁ = layerₖ(xₖ) | different per layer | fixed by you |
| ResNet | xₖ₊₁ = xₖ + Fₖ(xₖ) | different per block | fixed by you |
| Neural ODE | x(t+h) ≈ x(t) + h·f(x, t) | **same f every time** | **chosen by the solver** |

**RK4 = 4 calls of the same f per step:**
- k₁ = f(x, t)
- k₂ = f(x + (h/2)k₁, t + h/2)
- k₃ = f(x + (h/2)k₂, t + h/2)
- k₄ = f(x + h·k₃, t + h)
- x(t + h) = x + (h/6)(k₁ + 2k₂ + 2k₃ + k₄)

**Concrete count** (3-layer MLP, input 64 + t, hidden 64; 50 RK4 steps):
- 50 × 4 = **200 calls of f** → 600 layer passes
- Parameters: **12,544**, shared across all calls
- 600 independent layers would need ≈ **2.5 million** parameters

**Adaptive solvers** (e.g. `dopri5`) pick step sizes from an error estimate → NFE (number of function evaluations) depends on the input and grows with training complexity. Track it.

`odeint` stores activations of all 200 calls; `odeint_adjoint` keeps one call's worth at a time.

In [ ]:
import numpy as np

d, steps, evals_per_step, n_layers = 64, 50, 4, 3
params = (d + 1) * d + d + 2 * (d * d + d)   # 3-layer MLP: (d+1)*d + 2*(d*d+d)
calls = steps * evals_per_step

print("calls of f                  :", calls)
print("layer passes                :", calls * n_layers)
print("shared parameters           :", params)
print("if every call had own weights:", calls * params)

# Verify RK4 on the toy model
f_toy = lambda x, t, w=0.5: w * x
def rk4_step(x, t, h):
    k1 = f_toy(x, t)
    k2 = f_toy(x + h/2*k1, t + h/2)
    k3 = f_toy(x + h/2*k2, t + h/2)
    k4 = f_toy(x + h*k3,   t + h)
    return x + h/6 * (k1 + 2*k2 + 2*k3 + k4)

x = 1.0
for i in range(10):
    x = rk4_step(x, i * 0.1, 0.1)
print(f"\nRK4, 10 steps: x(1) = {x:.6f}  exact = {np.exp(0.5):.6f}")

## 14. Training loop vocabulary: iteration, batch, epoch

Per **iteration**: forward solve with current θ → loss → backward solve for dL/dθ → optimizer update → next iteration uses the **new** θ.

- Within one forward/backward pass θ is frozen (all f calls use the same weights).
- θ changes **after every batch**, not once per epoch. So f is a different function each iteration, and an adaptive solver may choose different steps.
- Two kinds of derivative: **dx/dt** (what f returns, used by the solver) vs **dL/dθ** (from the backward pass, used by Adam/SGD).

| Term | Meaning |
|---|---|
| batch | small group of samples processed together |
| iteration | one forward + loss + backward + update on one batch |
| epoch | enough iterations to see the whole dataset once |

**iterations per epoch = dataset size / batch size** (e.g. 10,000 / 100 = 100 updates per epoch).

In [ ]:
import numpy as np

# Toy training run: find w so that x(1) = 2, starting from w = 0.5
# Model: dx/dt = w*x, x(0)=1 → x(1) = exp(w). Target: exp(w) = 2 → w* = ln(2) ≈ 0.6931

w, lr = 0.5, 0.1
for it in range(1, 6):
    x1    = np.exp(w)                    # forward solve (exact for this model)
    L     = (x1 - 2) ** 2
    dL_dw = 2 * (x1 - 2) * x1           # dL/dx1 * dx1/dw; dx1/dw = x1 comes from the adjoint
    print(f"iter {it}: w={w:.4f}  x(1)={x1:.4f}  L={L:.4f}  dL/dw={dL_dw:.4f}")
    w -= lr * dL_dw

print(f"\ntarget w = ln 2 = {np.log(2):.4f}")

## 15. Practical notes for a neural ODE project

- **Structure:** encoder → latent ODE block → decoder. The encoder/decoder are ordinary layers (backprop); the ODE block is where solver steps pile up.
- **Start with `odeint`** (backprop through the solver: exact, faster). Switch to **`odeint_adjoint`** only if memory is the bottleneck.
- **Dissipative latent dynamics** (common in fluid models) → backward reconstruction may be unstable. Before trusting the adjoint, compare gradients from both on a small batch. If they differ significantly, tighten `rtol`/`atol` or switch to a checkpointed adjoint.
- **Track NFE** during training; rising NFE means f is learning sharper dynamics and the solver is taking smaller steps.
- **Losses on several frames** → λ jumps at each observation time (handled automatically by the libraries).

Gradient comparison snippet (needs `torch` and `torchdiffeq`):

In [ ]:
try:
    import torch
    from torchdiffeq import odeint, odeint_adjoint

    class F(torch.nn.Module):
        def __init__(self, d=8):
            super().__init__()
            self.net = torch.nn.Sequential(
                torch.nn.Linear(d, 32), torch.nn.Tanh(), torch.nn.Linear(32, d))
        def forward(self, t, x):
            return self.net(x)

    torch.manual_seed(0)
    f = F()
    x0 = torch.randn(4, 8)
    t  = torch.tensor([0.0, 1.0])

    def grads(solver):
        f.zero_grad()
        xT = solver(f, x0, t, rtol=1e-6, atol=1e-8)[-1]
        xT.pow(2).sum().backward()
        return torch.cat([p.grad.flatten() for p in f.parameters()])

    g_bp, g_adj = grads(odeint), grads(odeint_adjoint)
    rel = (g_bp - g_adj).norm() / g_bp.norm()
    print(f"relative difference between backprop and adjoint gradients: {rel:.2e}")
    print("(small = adjoint is reliable for this model)")

except ImportError:
    print("torch / torchdiffeq not installed — snippet skipped (pip install torch torchdiffeq)")

## 16. Quick self-test

<details><summary>Does Adam need the chain rule?</summary>No. Adam consumes a gradient; the chain rule (backprop / adjoint) computes it.</details>

<details><summary>What does λ(t) mean?</summary>∂L/∂x(t): how much the loss changes if the state at time t is nudged.</details>

<details><summary>Why is adjoint memory O(1) in number of steps?</summary>Only the end state is stored; earlier states are rebuilt by integrating backward, and each VJP's autograd graph is freed after its step.</details>

<details><summary>Where does the minus sign in dλ/dt = −λ·∂f/∂x come from?</summary>The chain rule links λ(t) to λ(t+ε) ("earlier = later + something"); writing it as d/dt ("later − earlier") flips the sign. ∂L/∂f is not −λ.</details>

<details><summary>Is ∂f/∂x the acceleration?</summary>No. It compares speeds of nearby states at the same instant. Acceleration = ∂f/∂x · f.</details>

<details><summary>Gradient formula?</summary>dL/dw = ∫ λ · ∂f/∂w dt. (λ · f is the Hamiltonian = dL/dT.)</details>

<details><summary>Why no separate acceleration path?</summary>w only appears in the speed; downstream knock-on effects are carried by λ. For second-order models, add velocity to the state.</details>

<details><summary>When does the adjoint fail?</summary>Dissipative/stiff dynamics (backward integration amplifies errors), discontinuities, or when an approximate gradient isn't good enough.</details>

<details><summary>Is ReLU a problem for neural ODEs?</summary>No. You invert the step x + h·f(x), never f itself.</details>

<details><summary>What makes states rebuildable?</summary>Same size throughout + residual (additive) updates + small/controlled changes.</details>

<details><summary>How many parameters does a neural ODE with a 3-layer MLP and 200 f calls have?</summary>Just the MLP's (12,544 in the example); weights are shared across all calls.</details>

<details><summary>Iterations per epoch?</summary>Dataset size / batch size. Weights update every iteration.</details>